In [ ]:
!pip install python-terrier --quiet
!pip install nltk --quiet
!pip install --ignore-installed blinker
!pip install git+https://github.com/experimaestro/experimaestro-ir.git --quiet
!pip install transformers
!pip install flair
!git clone https://github.com/terrierteam/terrier-prf/
!apt-get install maven   #used for Java projects to manage project dependencies and build processes
%cd /content/terrier-prf/
!mvn install
!pwd
%cd ..

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.7/110.7 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 11.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.9/337.9 kB 13.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.4/287.4 kB 12.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.1 MB/s eta 0:00:00
  Preparing metadata (setup.

In [ ]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import *
from nltk.stem.porter import *
from flair.data import Sentence
from flair.embeddings import WordEmbeddings
from transformers import AutoTokenizer, AutoModel
import tensorflow as tf
import tensorflow_hub as hub
import gensim
from gensim.models import Word2Vec
pd.set_option('display.max_colwidth', 150)
import pyterrier as pt
if not pt.started():
  # In this lab, we need to specify that we start PyTerrier with PRF enabled
  pt.init(boot_packages=["com.github.terrierteam:terrier-prf:-SNAPSHOT"])


terrier-assemblies 5.9 jar-with-dependencies not found, downloading to /root/.pyterrier...
Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...
Done
terrier-prf -SNAPSHOT jar not found, downloading to /root/.pyterrier...
Done


PyTerrier 0.10.1 has loaded Terrier 5.9 (built by craigm on 2024-05-02 17:40) and terrier-helper 0.0.8



## **1) Data Collection**

In [ ]:
df = pd.read_csv("/content/tweet_emotions.csv")
df

,tweet_id,sentiment,content
0,1956967341,empty,@tiffanylue i know i was listenin to bad habit earlier and i started freakin at his part =[
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin on your call...
2,1956967696,sadness,Funeral ceremony...gloomy friday...
3,1956967789,enthusiasm,wants to hang out with friends SOON!
4,1956968416,neutral,"@dannycastillo We want to trade with someone who has Houston tickets, but no one will."
...,...,...,...
39995,1753918954,neutral,@JohnLloydTaylor
39996,1753919001,love,Happy Mothers Day All my love
39997,1753919005,love,"Happy Mother's Day to all the mommies out there, be you woman or man as long as you're 'momma' to someone this is your day!"
39998,1753919043,happiness,@niariley WASSUP BEAUTIFUL!!! FOLLOW ME!! PEEP OUT MY NEW HIT SINGLES WWW.MYSPACE.COM/IPSOHOT I DEF. WAT U IN THE VIDEO!!


## **2) Preprocessing**

In [ ]:
nltk.download('stopwords')
print(stopwords.words('english'))
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

True

In [ ]:
stop_words = set(stopwords.words('english'))

In [ ]:
# Function to remove stopwords
def remove_stopwords(text):

    tokens = word_tokenize(text)
    filtered_tokens = [word.lower() for word in tokens if word.lower() not in stop_words] #Lower is used to normalize al the words make them in lower case
    print('Tokens are:',tokens,'\n')
    return ' '.join(filtered_tokens)

# Apply the remove_stopwords function to the 'content' column
df['processed_text'] = df['content'].apply(remove_stopwords)

Streaming output truncated to the last 5000 lines.
Tokens are: ['@', 'aminorjourney', '-', 'I', 'did', 'see', 'it', '...', 'looks', 'so', 'retro', '!', 'I', 'would', "n't", 'say', 'no', 'to', 'rerecordings', '...', 'fans', 'LOVED', 'Given', 'One', 'Change', '.', 'You', 'have', 'CS', 'Fans', '!'] 

Tokens are: ['@', 'kjofficial', 'I', "'m", 'sure', 'you', 'left', 'the', 'audience', 'awestruck', 'Katherine', '.', 'Looking', 'forward', 'to', 'reading', 'some', 'wonderful', 'reports', '.'] 

Tokens are: ['@', 'anjelfich', 'yay', '!', 'You', "'re", 'on', 'twitter', '!'] 

Tokens are: ['Found', 'controls', 'for', 'left', 'hand', 'people', 'like', 'me', 'on', 'twitterrific', '.', 'Excellent'] 

Tokens are: ['@', 'bradiewebbstack', 'i', 'had', 'had', 'a', 'baked', 'dinner', 'yummy', 'cant', 'wait', 'for', 'new', 'short', 'stack', 'tv', ',', 'what', 'kind', 'of', 'dips', 'shall', 'it', 'be', '?'] 

Tokens are: ['@', 'bradiewebbstack', 'haha', 'sounds', 'like', 'your', 'going', 'to', 'have', 'he

In [ ]:
print('dataFrame after processing:\n')
df

dataFrame after processing:



,tweet_id,sentiment,content,processed_text
0,1956967341,empty,@tiffanylue i know i was listenin to bad habit earlier and i started freakin at his part =[,@ tiffanylue know listenin bad habit earlier started freakin part = [
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin on your call...,layin n bed headache ughhhh ... waitin call ...
2,1956967696,sadness,Funeral ceremony...gloomy friday...,funeral ceremony ... gloomy friday ...
3,1956967789,enthusiasm,wants to hang out with friends SOON!,wants hang friends soon !
4,1956968416,neutral,"@dannycastillo We want to trade with someone who has Houston tickets, but no one will.","@ dannycastillo want trade someone houston tickets , one ."
...,...,...,...,...
39995,1753918954,neutral,@JohnLloydTaylor,@ johnlloydtaylor
39996,1753919001,love,Happy Mothers Day All my love,happy mothers day love
39997,1753919005,love,"Happy Mother's Day to all the mommies out there, be you woman or man as long as you're 'momma' to someone this is your day!","happy mother 's day mommies , woman man long 're 'momma ' someone day !"
39998,1753919043,happiness,@niariley WASSUP BEAUTIFUL!!! FOLLOW ME!! PEEP OUT MY NEW HIT SINGLES WWW.MYSPACE.COM/IPSOHOT I DEF. WAT U IN THE VIDEO!!,@ niariley wassup beautiful ! ! ! follow ! ! peep new hit singles www.myspace.com/ipsohot def . wat u video ! !


In [ ]:
stemmer = PorterStemmer()

In [ ]:
def Steem_text(text):

    tokens = word_tokenize(text)
    stemmed_tokens = [stemmer.stem(word) for word in tokens]
    return ' '.join(stemmed_tokens)

In [ ]:
#a function to clean the documents
def clean(text):
   text = re.sub(r"http\S+", " ", text) # remove urls
   text = re.sub(r"RT ", " ", text) # remove rt
   text = re.sub(r"@[\w]*", " ", text) # remove handles
   text = re.sub(r"[\.\,\#_\|\:\?\?\/\=]", " ", text) # remove special characters
   text = re.sub(r'\t', ' ', text) # remove tabs
   text = re.sub(r'\n', ' ', text) # remove line jump
   text = re.sub(r"\s+", " ", text) # remove extra white space
   text = text.strip()
   return text
df['processed_text'] = df['processed_text'].apply(clean)
df

,tweet_id,sentiment,content,processed_text
0,1956967341,empty,@tiffanylue i know i was listenin to bad habit earlier and i started freakin at his part =[,tiffanylue know listenin bad habit earlier started freakin part [
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin on your call...,layin n bed headache ughhhh waitin call
2,1956967696,sadness,Funeral ceremony...gloomy friday...,funeral ceremony gloomy friday
3,1956967789,enthusiasm,wants to hang out with friends SOON!,wants hang friends soon !
4,1956968416,neutral,"@dannycastillo We want to trade with someone who has Houston tickets, but no one will.",dannycastillo want trade someone houston tickets one
...,...,...,...,...
39995,1753918954,neutral,@JohnLloydTaylor,johnlloydtaylor
39996,1753919001,love,Happy Mothers Day All my love,happy mothers day love
39997,1753919005,love,"Happy Mother's Day to all the mommies out there, be you woman or man as long as you're 'momma' to someone this is your day!",happy mother 's day mommies woman man long 're 'momma ' someone day !
39998,1753919043,happiness,@niariley WASSUP BEAUTIFUL!!! FOLLOW ME!! PEEP OUT MY NEW HIT SINGLES WWW.MYSPACE.COM/IPSOHOT I DEF. WAT U IN THE VIDEO!!,niariley wassup beautiful ! ! ! follow ! ! peep new hit singles www myspace com ipsohot def wat u video ! !


In [ ]:
df['processed_text']= df['processed_text'].apply(Steem_text)
df

,tweet_id,sentiment,content,processed_text
0,1956967341,empty,@tiffanylue i know i was listenin to bad habit earlier and i started freakin at his part =[,tiffanylu know listenin bad habit earlier start freakin part [
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin on your call...,layin n bed headach ughhhh waitin call
2,1956967696,sadness,Funeral ceremony...gloomy friday...,funer ceremoni gloomi friday
3,1956967789,enthusiasm,wants to hang out with friends SOON!,want hang friend soon !
4,1956968416,neutral,"@dannycastillo We want to trade with someone who has Houston tickets, but no one will.",dannycastillo want trade someon houston ticket one
...,...,...,...,...
39995,1753918954,neutral,@JohnLloydTaylor,johnlloydtaylor
39996,1753919001,love,Happy Mothers Day All my love,happi mother day love
39997,1753919005,love,"Happy Mother's Day to all the mommies out there, be you woman or man as long as you're 'momma' to someone this is your day!",happi mother 's day mommi woman man long 're 'momma ' someon day !
39998,1753919043,happiness,@niariley WASSUP BEAUTIFUL!!! FOLLOW ME!! PEEP OUT MY NEW HIT SINGLES WWW.MYSPACE.COM/IPSOHOT I DEF. WAT U IN THE VIDEO!!,niariley wassup beauti ! ! ! follow ! ! peep new hit singl www myspac com ipsohot def wat u video ! !


In [ ]:
df['docno'] = df['tweet_id'].astype(str)
df

,tweet_id,sentiment,content,processed_text,docno
0,1956967341,empty,@tiffanylue i know i was listenin to bad habit earlier and i started freakin at his part =[,tiffanylu know listenin bad habit earlier start freakin part [,1956967341
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin on your call...,layin n bed headach ughhhh waitin call,1956967666
2,1956967696,sadness,Funeral ceremony...gloomy friday...,funer ceremoni gloomi friday,1956967696
3,1956967789,enthusiasm,wants to hang out with friends SOON!,want hang friend soon !,1956967789
4,1956968416,neutral,"@dannycastillo We want to trade with someone who has Houston tickets, but no one will.",dannycastillo want trade someon houston ticket one,1956968416
...,...,...,...,...,...
39995,1753918954,neutral,@JohnLloydTaylor,johnlloydtaylor,1753918954
39996,1753919001,love,Happy Mothers Day All my love,happi mother day love,1753919001
39997,1753919005,love,"Happy Mother's Day to all the mommies out there, be you woman or man as long as you're 'momma' to someone this is your day!",happi mother 's day mommi woman man long 're 'momma ' someon day !,1753919005
39998,1753919043,happiness,@niariley WASSUP BEAUTIFUL!!! FOLLOW ME!! PEEP OUT MY NEW HIT SINGLES WWW.MYSPACE.COM/IPSOHOT I DEF. WAT U IN THE VIDEO!!,niariley wassup beauti ! ! ! follow ! ! peep new hit singl www myspac com ipsohot def wat u video ! !,1753919043


## **3) Indexing**

In [ ]:
indexer = pt.DFIndexer("./DatasetIndex", overwrite=True)
index_ref = indexer.index(df["processed_text"], df["docno"])
print(index_ref.toString())

17:48:59.593 [main] WARN org.terrier.structures.indexing.Indexer - Adding an empty document to the index (1957091954) - further warnings are suppressed
17:49:30.190 [main] WARN org.terrier.structures.indexing.Indexer - Indexed 43 empty documents
./DatasetIndex/data.properties


In [ ]:
#we will first load the index
index = pt.IndexFactory.of(index_ref)
#we will call getCollectionStatistics() to check the stats
print(index.getCollectionStatistics().toString())

Number of documents: 40000
Number of terms: 39621
Number of postings: 277379
Number of fields: 0
Number of tokens: 285788
Field names: []
Positions:   false



In [ ]:
for kv in index.getLexicon():
  print("%s -> %s " % (kv.getKey(), kv.getValue().toString()))

Streaming output truncated to the last 5000 lines.
teeth -> term1271 Nt=34 TF=34 maxTF=1 @{0 499255 1} 
teetot -> term6683 Nt=2 TF=2 maxTF=1 @{0 499336 1} 
teev -> term38226 Nt=1 TF=1 maxTF=1 @{0 499341 3} 
teflon -> term29515 Nt=1 TF=1 maxTF=1 @{0 499345 3} 
teg -> term22408 Nt=1 TF=1 maxTF=1 @{0 499349 1} 
tegan -> term22705 Nt=1 TF=1 maxTF=1 @{0 499352 7} 
tegs03 -> term37779 Nt=1 TF=1 maxTF=1 @{0 499356 5} 
teh -> term7821 Nt=7 TF=7 maxTF=1 @{0 499360 5} 
tehblu -> term17059 Nt=1 TF=1 maxTF=1 @{0 499380 3} 
tehcheapon -> term15252 Nt=1 TF=1 maxTF=1 @{0 499383 7} 
tehe -> term1283 Nt=3 TF=3 maxTF=1 @{0 499387 3} 
teheh -> term38600 Nt=1 TF=1 maxTF=1 @{0 499396 7} 
tehr -> term13505 Nt=1 TF=1 maxTF=1 @{0 499400 7} 
tei -> term35556 Nt=1 TF=1 maxTF=1 @{0 499404 3} 
teifion -> term29025 Nt=1 TF=1 maxTF=1 @{0 499408 3} 
teignmouth -> term27384 Nt=1 TF=1 maxTF=1 @{0 499412 1} 
teiisha -> term34006 Nt=1 TF=1 maxTF=1 @{0 499415 7} 
tekenen -> term25412 Nt=1 TF=1 maxTF=1 @{0 499419 5} 
teks

In [ ]:
index.getLexicon()["result"].getDocumentFrequency()

33

In [ ]:
x = index.getDirectIndex()
y = index.getDocumentIndex()
z = index.getLexicon()
a = 0
for posting in x.getPostings(y.getDocumentEntry(a)):
    termid = posting.getId()
    B = z.getLexiconEntry(termid)
    print("%s with frequency %d" % (B.getKey(),posting.getFrequency()))

freakin with frequency 1
tiffanylu with frequency 1
earlier with frequency 1
start with frequency 1
habit with frequency 1
bad with frequency 1
know with frequency 1
part with frequency 1
listenin with frequency 1


## **4) Query processing**

In [ ]:
def preprocess(sentence):
    sentence = remove_stopwords(sentence)
    sentence = clean(sentence)
    sentence = Steem_text(sentence)
    return sentence

In [ ]:
# Preprocess user query
query = "empty"
query = preprocess(query)

Tokens are: ['empty'] 



In [ ]:
# Set up retrieval model with TF-IDF as weighting model
tfidf_retr = pt.BatchRetrieve(index, controls={"wmodel": "TF_IDF"}, num_results=10)

# Search for relevant documents using the retrieval model
results = tfidf_retr.search(query)
results

,qid,docid,docno,rank,score,query
0,1,8158,1962056440,0,7.888069,empti
1,1,13031,1963821958,1,7.694938,empti
2,1,18784,1966076425,2,7.295877,empti
3,1,26825,1695546329,3,7.295877,empti
4,1,8677,1962256777,4,6.786393,empti
5,1,13094,1963873810,5,6.786393,empti
6,1,332,1957041767,6,6.343420,empti
7,1,10456,1962910181,7,6.343420,empti
8,1,6509,1961284984,8,5.954733,empti
9,1,10100,1962822536,9,5.954733,empti


In [ ]:
# Retrieve top relevant documents based on the query
top_answer = df[df["docno"].isin(['1962056440', '1963821958'])]['processed_text']
top_answer

8158                                           room empti
13031    gut kitchen empti liter empti even kid 'm hungri
Name: processed_text, dtype: object

In [ ]:
# Calculate frequency of the term 'empty' in the top related documents
freq_of_items = []
for i in range(len(top_answer)):
    tweet = top_answer.iloc[i].split()
    counter = 0
    for word_in_tweet in tweet:
        if word_in_tweet == 'empty':
            counter += 1
    freq_of_items.append(counter)
print('The frequency of the item in the top related docs are:', freq_of_items)

The frequency of the item in the top related docs are: [0, 0]


In [ ]:
indexer = pt.DFIndexer("./DatasetIndex", overwrite=True)
# index the text, record the docnos as metadata
index_ref = indexer.index(df["processed_text"], df["docno"])
print(index_ref.toString())

17:50:27.605 [main] WARN org.terrier.structures.indexing.Indexer - Adding an empty document to the index (1957091954) - further warnings are suppressed
17:50:53.555 [main] WARN org.terrier.structures.indexing.Indexer - Indexed 43 empty documents
./DatasetIndex/data.properties


In [ ]:
index = pt.IndexFactory.of(index_ref)

## **5) Query Expantion**

In [ ]:
# Define our retrieval model
bm25 = pt.BatchRetrieve(index, wmodel="BM25",num_results=10)
results = bm25.search(query)
results

,qid,docid,docno,rank,score,query
0,1,8158,1962056440,0,14.428152,empti
1,1,13031,1963821958,1,14.074893,empti
2,1,18784,1966076425,2,13.344967,empti
3,1,26825,1695546329,3,13.344967,empti
4,1,8677,1962256777,4,12.413064,empti
5,1,13094,1963873810,5,12.413064,empti
6,1,332,1957041767,6,11.602818,empti
7,1,10456,1962910181,7,11.602818,empti
8,1,6509,1961284984,8,10.891867,empti
9,1,10100,1962822536,9,10.891867,empti


In [ ]:
df[['content']][df['docno'].isin(results['docno'].loc[0:4].tolist())]

,content
8158,Room is so empty
8677,i have a empty house and no ine to share it with
13031,Gutted. The kitchen is empty literally EMPTY. No even kidding. I'm so hungry
18784,back to Roseburg...and an empty apartment
26825,Excited about having an empty apartment to ourselves for a little while


In [ ]:
rm3_expander = pt.rewrite.RM3(index, fb_terms=10, fb_docs=100)

rm3_qe = bm25 >> rm3_expander
expanded_query = rm3_qe.search(query).iloc[0]["query"]

expanded_query

'applypipeline:off feel^0.022337455 empti^0.779976130 todai^0.022337455 hou^0.033506181 in^0.016753091 room^0.033506181 back^0.022337455 excit^0.022337455 littl^0.022337455 sad^0.024571197'

In [ ]:
# Just print the expanded query with term scores
for s in expanded_query.split()[1:]:
  print(s)

print("\n" + query)

feel^0.022337455
empti^0.779976130
todai^0.022337455
hou^0.033506181
in^0.016753091
room^0.033506181
back^0.022337455
excit^0.022337455
littl^0.022337455
sad^0.024571197

empti


In [ ]:
# After that you can search using the expanded query
expanded_query_formatted = ' '.join(expanded_query.split()[1:])

results_wqe = bm25.search(expanded_query_formatted)

print("   Before Expansion    After Expansion")
print(pd.concat([results[['docid','score']][0:5].add_suffix('_1'),
            results_wqe[['docid','score']][0:5].add_suffix('_2')], axis=1).fillna(''))


#Let's check the tweets text for the top 5 retrieved tweets
df[['content']][df['docno'].isin(results_wqe['docno'].loc[0:5].tolist())]

   Before Expansion    After Expansion
   docid_1    score_1  docid_2    score_2
0     8158  14.428152     8158  14.972125
1    13031  14.074893    13031  14.074893
2    18784  13.344967    26825  13.920685
3    26825  13.344967    18784  13.550158
4     8677  12.413064     8677  12.820628


,content
8158,Room is so empty
8677,i have a empty house and no ine to share it with
13031,Gutted. The kitchen is empty literally EMPTY. No even kidding. I'm so hungry
13094,Hmmmm where is everyone my house is empty. am all alone.
18784,back to Roseburg...and an empty apartment
26825,Excited about having an empty apartment to ourselves for a little while


## **6) User Interface**

In [ ]:
!pip install flask_ngrok
!pip install pyngrok

In [ ]:
df2 = df.head(50)

df2 = df2.to_dict()

df2

{'tweet_id': {0: 1956967341,
  1: 1956967666,
  2: 1956967696,
  3: 1956967789,
  4: 1956968416,
  5: 1956968477,
  6: 1956968487,
  7: 1956968636,
  8: 1956969035,
  9: 1956969172,
  10: 1956969456,
  11: 1956969531,
  12: 1956970047,
  13: 1956970424,
  14: 1956970860,
  15: 1956971077,
  16: 1956971170,
  17: 1956971206,
  18: 1956971473,
  19: 1956971586,
  20: 1956971981,
  21: 1956972097,
  22: 1956972116,
  23: 1956972270,
  24: 1956972359,
  25: 1956972444,
  26: 1956972557,
  27: 1956972884,
  28: 1956973598,
  29: 1956973690,
  30: 1956974706,
  31: 1956975441,
  32: 1956975860,
  33: 1956975876,
  34: 1956975927,
  35: 1956976187,
  36: 1956976312,
  37: 1956976371,
  38: 1956976557,
  39: 1956976681,
  40: 1956977084,
  41: 1956977187,
  42: 1956977618,
  43: 1956977624,
  44: 1956978276,
  45: 1956978410,
  46: 1956978668,
  47: 1956979150,
  48: 1956979437,
  49: 1956979756},
 'sentiment': {0: 'empty',
  1: 'sadness',
  2: 'sadness',
  3: 'enthusiasm',
  4: 'neutral',
  5

In [ ]:
def sui(df2 , que):
 i = 0

 quer = preprocess(que)

 docs_id = []

 for key, value in df2.items():
   if key == 'processed_text':
         val = value.values()
         for doc in val:
           terms = doc.split()
           for term in terms:
             if term == quer and i not in docs_id:
               docs_id.append(f'''Document number {i} -----> \n{df["content"][i]}''')
           i = i + 1
 return docs_id

In [ ]:
query2 = "bad"
X = sui(df2 , query2)
X

Tokens are: ['bad'] 



['Document number 0 -----> \n@tiffanylue i know  i was listenin to bad habit earlier and i started freakin at his part =[',
 "Document number 20 -----> \n@raaaaaaek oh too bad! I hope it gets better. I've been having sleep issues lately too"]

In [ ]:
from google.colab.output import eval_js
print (eval_js("google.colab.kernel.proxyPort(5000)"))

https://61rau1xr3k5-496ff2e9c6d22116-5000-colab.googleusercontent.com/


In [ ]:
from flask import Flask, request
from flask_ngrok import run_with_ngrok

# Assuming you've already defined the sui function and imported necessary modules

app = Flask(__name__)
run_with_ngrok(app)

@app.route("/")
def home():
    return """
    <style>
        body {
            background-color: 	#78a1d1;
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 0;
        }

        .header {
            background-color:	#78a1d1 ;
            color: #403072 ;
            padding: 20px 0;

        }

        .container {
            text-align: center;
        }

        h1 {
            text-align: center;
            margin:0;
            padding: 10px 0;
        }

        #searchInput {
            padding: 10px;
            border: 1px solid #ccc;
            border-radius: 50px; /* Increased border-radius for a rounded appearance */
            margin-bottom: 10px;
            width: 300px; /* Adjust the width as needed */
            box-sizing: border-box; /* Include padding and border in the element's total width */
            transition: border-color 0.5s; /* Smooth transition for border color change */
        }

        #searchInput:focus {
            border-color: #403072; /* Change border color on focus */
        }

        button {
            padding: 10px 20px;
            background-color: #403072;
            color: white;
            border: none;
            border-radius: 20px; /* Increased border-radius for a rounded appearance */
            cursor: pointer;
            transition: background-color 0.3s; /* Smooth transition for background color change */
        }

        button:hover {
            background-color: #7d73af; /* Change background color on hover */
        }

          #searchResult {
            text-align: 10 px; /* Center the search results */
            color: #403072; /* Set the color of search results */
            margin: 10 px;
            padding: 10px
        }
    </style>

    <div class="header">
        <h1>Search Engine</h1>
    </div>
    <div class="container">
        <input type="text" id="searchInput" placeholder="search...">
        <button onclick="search()">Search</button>
    </div>
    <div id="searchResult"></div>

    <script>
        function search() {
            var searchTerm = document.getElementById("searchInput").value;
            fetch('/search', {
                method: 'POST',
                body: JSON.stringify({ query: searchTerm }),
                headers:{
                    'Content-Type': 'application/json'
                }
            })
            .then(response => response.json())
            .then(data => {
                console.log("Received data:", data); // Debug: Check if data is received
                var resultDiv = document.getElementById("searchResult");
                resultDiv.innerHTML = "<h2>Relevant Documents IDs:</h2>";
                if (data.results.length === 0) {
                    resultDiv.innerHTML += "<p>No documents found</p>";
                } else {
                    data.results.forEach(doc => {
                        console.log("Displaying document:", doc); // Debug: Check if document is displayed
                        resultDiv.innerHTML += "<p>" + doc + "</p>";
                    });
                }
            })
            .catch(error => {
                console.error('Error occurred during fetch:', error); // Debug: Log fetch errors
            });
        }
    </script>
    """

@app.route("/search", methods=['POST'])
def search():
    query = request.json['query']
    print("Received query:", query)  # Debug: Check if Flask receives the query
    results = sui(df2, query)
    print("Search results:", results)  # Debug: Check if sui function returns results
    return {'results': results}

app.run()


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [11/May/2024 18:16:02] "GET / HTTP/1.1" 200 -
Exception in thread Thread-102:
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/urllib3/connection.py", line 174, in _new_conn
    conn = connection.create_connection(
  File "/usr/local/lib/python3.10/dist-packages/urllib3/util/connection.py", line 95, in create_connection
    raise err
  File "/usr/local/lib/python3.10/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 111] Connection refused

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/urllib3/connectionpool.py", line 715, in urlopen
    httplib_r

Received query: happi
Tokens are: ['happi'] 

Search results: []


INFO:werkzeug:127.0.0.1 - - [11/May/2024 18:16:30] "POST /search HTTP/1.1" 200 -


Received query: empti
Tokens are: ['empti'] 

Search results: []


...

INFO:werkzeug:127.0.0.1 - - [11/May/2024 18:49:14] "POST /search HTTP/1.1" 200 -


Received query: hungry
Tokens are: ['hungry'] 

Search results: []


INFO:werkzeug:127.0.0.1 - - [11/May/2024 19:26:35] "POST /search HTTP/1.1" 200 -


Received query: feel
Tokens are: ['feel'] 

Search results: ['Document number 25 -----> \nOn my way home n having 2 deal w underage girls drinking gin on da bus while talking bout keggers......damn i feel old', 'Document number 33 -----> \nfeels strong contractions but wants to go out.  http://plurk.com/p/wxidk', "Document number 36 -----> \n@ether_radio yeah :S i feel all funny cause i haven't slept enough  i woke my mum up cause i was singing she's not impressed :S you?", "Document number 43 -----> \nWhy do I have the feeling I should be packing and hitting for SFO around this time of the year? I think I'm missing something...", "Document number 45 -----> \nBed!!!!!... its time,..... hope i go to school tomorrow, all though i don't feel very well right now"]


INFO:werkzeug:127.0.0.1 - - [11/May/2024 19:26:48] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2024 19:26:49] "GET /favicon.ico HTTP/1.1" 404 -


## **7) Evaluation**

In [ ]:
vaswani_dataset = pt.datasets.get_dataset("vaswani")

data = vaswani_dataset.get_topics()

data['docno'] = data.index

# Rename column 'A' to 'X'
data = data.rename(columns={'query': 'Text'})

qrels = vaswani_dataset.get_qrels()

qrels['docno']=qrels['docno'].astype(str)

data

,qid,Text,docno
0,1,measurement of dielectric constant of liquids by the use of microwave techniques,0
1,2,mathematical analysis and design details of waveguide fed microwave radiations,1
2,3,use of digital computers in the design of band pass filters having given phase and attenuation characteristics,2
3,4,systems of data coding for information transfer,3
4,5,use of programs in engineering testing of computers,4
...,...,...,...
88,89,tunnel diode construction and its electrical characteristics explained,88
89,90,electronic density of states at the surface of a semiconductor compared with that at depth,89
90,91,resistivity of metallic thin films related to surface roughness,90
91,92,the phenomenon of radiation caused by charged particles moving in varying electric and magnetic fields,91


In [ ]:
indexref2 = vaswani_dataset.get_index()
index2 = pt.IndexFactory.of(indexref2)

print(index2.getCollectionStatistics().toString())

data.direct.bf:   0%|          | 0.00/388k [00:00<?, ?iB/s]

data.document.fsarrayfile:   0%|          | 0.00/234k [00:00<?, ?iB/s]

data.inverted.bf:   0%|          | 0.00/362k [00:00<?, ?iB/s]

data.lexicon.fsomapfile:   0%|          | 0.00/682k [00:00<?, ?iB/s]

data.lexicon.fsomaphash:   0%|          | 0.00/777 [00:00<?, ?iB/s]

...

data.meta.idx:   0%|          | 0.00/89.3k [00:00<?, ?iB/s]

data.meta.zdata:   0%|          | 0.00/224k [00:00<?, ?iB/s]

data.properties:   0%|          | 0.00/4.29k [00:00<?, ?iB/s]

md5sums:   0%|          | 0.00/619 [00:00<?, ?iB/s]

Number of documents: 11429
Number of terms: 7756
Number of postings: 224573
Number of fields: 1
Number of tokens: 271581
Field names: [text]
Positions:   false



In [ ]:
retr = pt.BatchRetrieve(index2, controls = {"wmodel": "TF_IDF"})

res = retr.search("mathematical")
res

,qid,docid,docno,rank,score,query
0,1,4746,4747,0,5.168347,mathematical
1,1,7399,7400,1,5.036916,mathematical
2,1,5629,5630,2,4.912003,mathematical
3,1,7997,7998,3,4.912003,mathematical
4,1,4546,4547,4,4.679886,mathematical
...,...,...,...,...,...,...
147,1,3484,3485,147,1.828498,mathematical
148,1,7283,7284,148,1.747822,mathematical
149,1,6714,6715,149,1.702745,mathematical
150,1,8622,8623,150,1.606095,mathematical


In [ ]:
eval = pt.Evaluate(res,qrels)
eval

{'map': 4.7960250544348844e-06, 'ndcg': 0.00022891881462746983}